# 01 · Train every model the tracking dataset needs, in one run

This one notebook trains, checks and installs both models behind `tracking.parquet` (court x, y
of every on-court player and the ball, 10 times a second), then shows what changed in the dataset
itself. **Run all** from top to bottom; after a disconnect, run all again and finished or
half-finished training runs pick up where they stopped (they are saved in Drive).

| Part | Model | What "better" means for the tracking dataset |
|---|---|---|
| A | detector: `Player`, `Ref`, `Ball`, `Hoop` | more players found per frame (recall is the main weakness: ~6 of 10 on `lebron_block`), tracks that stay on one player (IDF1), fewer phantom balls |
| B | court keypoints: 18 landmarks | more broadcast frames get an image-to-court mapping (no mapping = no court x, y) |
| C | end-to-end check | the pipeline on your own clips, before vs after: players placed per 10 Hz sample, share of samples with all ten, share with the ball |

What makes this tuned for tracking, not just for box scores:

* **Scores match the pipeline**: per-frame player recall at the pipeline's thresholds, the share of
  frames where every player was found, MOTA and IDF1 through the pipeline's own camera-compensated
  tracker on SportsMOT games no model trained on, and mapping coverage on held-back broadcast videos.
* **Thresholds are tuned for tracking**: the player threshold is picked by IDF1 on held-out
  validation SportsMOT clips (the tracker uses low-confidence boxes to keep a player's track alive
  through screens), and the ball threshold by F1 on validation frames. Both are saved to
  `models/tuned_config.yaml`, which notebook 00 reads.
* **Leakage-free**: every dataset's own random split is ignored; images from the same video,
  SportsMOT game or near-identical frame (across datasets) always share a split; anything the
  current models may have seen is left out of the comparison that decides.
* **A model is only installed if it beats the current one** on that comparison. The old file is
  kept as `*_prev.pt`.

Time on an A100: about 5 to 7 hours for the detector and 1 to 2 for the court model (L4 about
twice that). The first run also cuts SportsMOT down to its basketball clips (about 20 minutes).

In [ ]:
# Runtime > Change runtime type > GPU before running (A100 or L4 if you have them; T4 works, slower).
!nvidia-smi -L
from google.colab import drive, userdata
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"   # everything that must survive a disconnect lives here
for d in ["models", "runs", "reports", "training_data", "input_videos", "output"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
BRANCH = "main"   # to test a pull request, put its branch name here, then Runtime > Restart session > Run all
!rm -rf /content/courtvision && git clone -q -b {BRANCH} {REPO_URL} /content/courtvision
!pip -q install -e "/content/courtvision[models,train]"
import sys
sys.path.insert(0, "/content/courtvision")   # importable now, without restarting the session

## Settings

Change `RUN_TAG` whenever the data changes (new own labels, a new dataset version), so finished
runs from before are not reused.

In [ ]:
RUN_TAG = "v2"
TRAIN_DETECTOR = True
TRAIN_COURT = True
RUN_CROSS_VALIDATION = False   # leave-one-dataset-out: +2 to 3 h; says which dataset matters, does not change the final models
DELETE_SPORTSMOT_TARS = True   # once the basketball zip is safely in Drive, move the raw SportsMOT tars to Drive's trash
E2E_SECONDS, E2E_MAX_CLIPS = 60, 3   # part C: first minute of up to 3 clips in MyDrive/courtvision/input_videos

DETECTOR = dict(base="yolo11m.pt", imgsz=1280, epochs=150, patience=30)
COURT = dict(base="yolo11m-pose.pt", imgsz=640, epochs=200, patience=40)

# The models and thresholds in place before this run, kept for the before/after check in part C.
import shutil, yaml
from courtvision.config import Config
MODELS = f"{DRIVE_ROOT}/models"
CURRENT_DET, CURRENT_COURT = f"{MODELS}/player_detector.pt", f"{MODELS}/court_keypoints.pt"
TUNED = f"{MODELS}/tuned_config.yaml"   # thresholds picked below; notebook 00 reads it
BEFORE = "/content/before"
os.makedirs(BEFORE, exist_ok=True)
for f in [CURRENT_DET, CURRENT_COURT, TUNED]:
    if os.path.exists(f) and not os.path.exists(f"{BEFORE}/{os.path.basename(f)}"):
        shutil.copy(f, BEFORE)

def load_cfg(tuned=None):
    """Pipeline settings: configs/default.yaml, then a tuned_config.yaml if given and present."""
    extra = yaml.safe_load(open(tuned)) if tuned and os.path.exists(tuned) else {}
    return Config.load("/content/courtvision/configs/default.yaml", **(extra or {}))

CFG = load_cfg(TUNED)
print("current thresholds: players", CFG.detection.conf, "ball", CFG.detection.ball_conf)

def train(name, data_yaml, base, **args):
    """Train, or resume after a disconnect; returns the best weights. Finished runs are reused."""
    from ultralytics import YOLO
    run = f"{DRIVE_ROOT}/runs/{name}"
    if not os.path.exists(f"{run}/done"):
        last = f"{run}/weights/last.pt"
        if os.path.exists(last):
            try:
                YOLO(last).train(resume=True)
            except AssertionError as e:   # it had finished; only the marker was missing
                print(e)
        else:
            YOLO(base).train(data=data_yaml, project=f"{DRIVE_ROOT}/runs", name=name, exist_ok=True, **args)
        open(f"{run}/done", "w").close()
    return f"{run}/weights/best.pt"

def install(new, current, label):
    """Put the new weights in place, keeping the old file as *_prev.pt."""
    if os.path.exists(current):
        shutil.copy(current, current.replace(".pt", "_prev.pt"))
    shutil.copy(new, current)
    print(f"Installed the new {label}: {current}")

## 1 · Download the datasets

Keys come from Colab **Secrets** (key icon on the left): `ROBOFLOW_API_KEY`, and `HF_TOKEN` for
SportsMOT. Never paste a key into a cell. `v17` and `reloc2` are pinned so results stay
comparable; the two broadcast sets take their latest version, which is printed.

| Source | Used by | What it adds |
|---|---|---|
| `v17` | detector | your Roboflow set (workspace-5ujvu v17): all four classes, mostly full-court frames |
| `gamedet` | both | Basketball Game Detection (deeplearning-gmyc6): NBA broadcast frames |
| `broadcast` | both | Basketball Broadcast (basketball-w376e): broadcast frames |
| `reloc2` | court | hand-labelled court keypoints (fyp-3bwmg/reloc2-den7l v1), whole court in frame |
| `sportsmot` | detector | SportsMOT basketball clips with every player boxed and tracked: crowds, overlaps, blur |
| `own`, `own_court` | both | your frames from notebook 02, corrected in Roboflow (weighted x3) |
| `own_test`, `own_court_test` | both | frames of clips you keep back: scored on, never trained on |
| `negatives` | detector | crowd, bench, replays, ads: nothing to box |

Anything missing is skipped.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))

def download(workspace, project, version=None):
    """A Roboflow dataset in YOLO format; version None means the latest."""
    p = rf.workspace(workspace).project(project)
    version = version or max(int(str(v.version).split("/")[-1]) for v in p.versions())
    d = p.version(version).download("yolov8", location=f"/content/datasets/{project}-v{version}")
    print(f"{workspace}/{project} v{version} -> {d.location}")
    return d.location

ROOTS = {"gamedet": download("deeplearning-gmyc6", "basketball-game-detection"),
         "broadcast": download("basketball-w376e", "basketball-broadcast")}
if TRAIN_DETECTOR:
    ROOTS["v17"] = download("workspace-5ujvu", "basketball-players-fy4c2-vfsuv", 17)
if TRAIN_COURT:
    ROOTS["reloc2"] = download("fyp-3bwmg", "reloc2-den7l", 1)

### SportsMOT basketball

SportsMOT (CC BY-NC 4.0, gated on Hugging Face) has three sports and a test split without labels.
Only its **basketball train and val clips** are used, kept in Drive as one zip,
`training_data/sportsmot_basketball.zip` (a few GB). The first run makes that zip:

1. from the tars already in `training_data/sportsmot/dataset/` (`train.tar`, `val.tar`; `test.tar`
   is never read), or, if there are none, from Hugging Face without the test split;
2. the clip list comes from `splits_txt/basketball.txt` wherever it sits (the loose folder next to
   `dataset/` is fine), or from the copy built into `courtvision/training/sportsmot.py`;
3. the zip is copied to Drive and checked (it opens and holds every clip); only then, with
   `DELETE_SPORTSMOT_TARS`, the raw tars are moved to Drive's trash. **Empty the trash** to get
   the space back: files in the trash still count against your storage.

Every later run just unpacks the zip to Colab's disk, which is also much faster to read than Drive.

In [ ]:
import glob, json, zipfile
from pathlib import Path
from courtvision.training import sportsmot

SM_RAW = f"{DRIVE_ROOT}/training_data/sportsmot"            # raw download in Drive (tars), if any
SM_ZIP = f"{DRIVE_ROOT}/training_data/sportsmot_basketball.zip"
SM_INFO = f"{DRIVE_ROOT}/training_data/sportsmot_basketball.json"   # written once the Drive zip is checked
LOCAL_ZIP = "/content/sportsmot_basketball.zip"
gb = lambda n: round(n / 1e9, 2)

def zip_ok(path):
    try:
        with zipfile.ZipFile(path) as z:
            return "splits_txt/basketball.txt" in z.namelist()
    except (zipfile.BadZipFile, OSError):
        return False

if not zip_ok(SM_ZIP) and not zip_ok(LOCAL_ZIP):
    try:
        raw = SM_RAW if sportsmot.archives(SM_RAW) or glob.glob(f"{SM_RAW}/**/img1", recursive=True) else None
        if raw is None:   # nothing in Drive: download train + val (no test split) to Colab's disk
            from huggingface_hub import snapshot_download
            raw = "/content/sportsmot_raw"
            snapshot_download("MCG-NJU/SportsMOT", repo_type="dataset", token=userdata.get("HF_TOKEN"),
                              local_dir=raw, ignore_patterns=["*test*"])
        print("reading", [a.name for a in sportsmot.archives(raw)] or raw)
        info = sportsmot.basketball_zip(raw, LOCAL_ZIP)
        print(info)
        assert info["sequences"]["train"] > 0, "no basketball clips found in the download"
        size = os.path.getsize(LOCAL_ZIP)
        if size + 1e9 < shutil.disk_usage(DRIVE_ROOT).free:
            shutil.copy(LOCAL_ZIP, SM_ZIP)
            if os.path.getsize(SM_ZIP) == size and sportsmot.check_zip(SM_ZIP, info["sequences"]):
                json.dump(info, open(SM_INFO, "w"))   # marks the Drive zip as complete
                print("basketball zip saved to Drive:", SM_ZIP)
            else:
                print("the copy to Drive is incomplete; this session uses the zip on Colab's disk")
        else:
            print(f"Drive has under {gb(size + 1e9)} GB free, so the zip stays on Colab's disk for this session only.\n"
                  "Empty Drive's trash and run this cell again to keep it.")
        if raw == "/content/sportsmot_raw":
            shutil.rmtree(raw)
    except Exception as e:
        print("SportsMOT not available:", repr(e))

# The raw tars go only once the Drive zip is known to hold every basketball clip.
tars = glob.glob(f"{SM_RAW}/**/*.tar*", recursive=True)
if DELETE_SPORTSMOT_TARS and tars and os.path.exists(SM_INFO) and sportsmot.check_zip(SM_ZIP, json.load(open(SM_INFO))["sequences"]):
    freed = sum(os.path.getsize(a) for a in tars)
    for a in tars:
        os.remove(a)   # on the Drive mount this moves the file to Drive's trash
    print(f"moved {gb(freed)} GB of SportsMOT tars to Drive's trash ({[Path(a).name for a in tars]}): empty the trash to free the space")

zip_path = SM_ZIP if zip_ok(SM_ZIP) else LOCAL_ZIP
SM_DATA = sportsmot.unpack(zip_path, "/content/sportsmot") if zip_ok(zip_path) else None
if SM_DATA:
    print({s: len(sportsmot.basketball_sequences(SM_DATA, s)) for s in ["train", "val"]}, "basketball clips")

# Part A · Detector

## A1 · SportsMOT frames as detector training data

SportsMOT boxes players only, so referees, the ball and the hoop are filled in from the current
detector's confident boxes, and a frame where it is unsure about one of them (a box between 0.15
and 0.5) is skipped rather than half-labelled. Those filled-in classes are never scored. One frame
in 10 is used (neighbouring frames are near-identical). Frames are named `game__clip_frame` so a
whole game stays on one side of every split.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from courtvision.training.labels import fill_missing
from courtvision.training.sources import DETECT_CLASSES, class_map

SM_YOLO = "/content/datasets/sportsmot-yolo"
if TRAIN_DETECTOR and SM_DATA and os.path.exists(CURRENT_DET) and not os.path.exists(f"{SM_YOLO}/data.yaml"):
    teacher = YOLO(CURRENT_DET)
    to_ours = class_map([teacher.names[i] for i in sorted(teacher.names)], DETECT_CLASSES)  # teacher id -> our id
    fill = [DETECT_CLASSES.index(c) for c in ["Ref", "Ball", "Hoop"]]

    def fill_from_teacher(img, players):
        b = teacher.predict(img, imgsz=1280, conf=0.15, verbose=False)[0].boxes
        cls = np.array([to_ours.get(int(c), -1) for c in b.cls.cpu().numpy()], int)
        return fill_missing(players, np.zeros(len(players), int), b.xyxy.cpu().numpy(), cls,
                            b.conf.cpu().numpy(), fill_classes=fill)

    print(sportsmot.to_yolo(SM_DATA, SM_YOLO, DETECT_CLASSES, stride=10, fill=fill_from_teacher))

## A2 · One leakage-free detector dataset

`weight` repeats a source's training images (never validation or test). The leakage checks must
all be 0. The table shows images and boxes per class for each source and split (under ~100 test
boxes of a class in a source gives a noisy score). `largest_component_share` above ~0.3 means that
source's splits are coarse.

In [ ]:
from courtvision.training.sources import Source, build, leakage_report, summary
from courtvision.training.look import show_grid

TD = f"{DRIVE_ROOT}/training_data"
DET_DATA = "/content/datasets/detector"
if TRAIN_DETECTOR:
    DET_SOURCES = [Source("v17", ROOTS["v17"]), Source("gamedet", ROOTS["gamedet"]), Source("broadcast", ROOTS["broadcast"])]
    if os.path.exists(f"{SM_YOLO}/data.yaml"):
        DET_SOURCES.append(Source("sportsmot", SM_YOLO, group_by="prefix"))
    if os.path.exists(f"{TD}/own_labelled/data.yaml"):
        DET_SOURCES.append(Source("own", f"{TD}/own_labelled", weight=3))
    if os.path.exists(f"{TD}/own_test/data.yaml"):
        DET_SOURCES.append(Source("own_test", f"{TD}/own_test", role="test"))
    if os.path.isdir(f"{TD}/negatives"):
        DET_SOURCES.append(Source("negatives", f"{TD}/negatives", role="negatives"))

    det_manifest = build(DET_SOURCES, DETECT_CLASSES, DET_DATA, val_frac=0.15, test_frac=0.15, seed=0)
    report = leakage_report(det_manifest)
    print(report)
    assert report["components_in_two_splits"] == report["groups_in_two_splits"] == report["test_source_images_not_in_test"] == 0
    det_manifest.to_csv(f"{DRIVE_ROOT}/reports/detector_manifest_{RUN_TAG}.csv", index=False)

    # Classes each source labels itself. SportsMOT's referees, balls and hoops came from the current detector.
    PSEUDO_CLASSES = {"sportsmot": ["Ref", "Ball", "Hoop"]}
    SCORED = {s: [c for c in DETECT_CLASSES if g[c].sum() > 0 and c not in PSEUDO_CLASSES.get(s, [])]
              for s, g in det_manifest.groupby("source") if s != "negatives"}
    print("scored classes:", SCORED)
    display(summary(det_manifest, DETECT_CLASSES))

In [ ]:
# Look at each source before training on it: every player on the court should be boxed. A dataset
# that skips players teaches the model that players are background. Saved to reports/look_<source>.jpg.
if TRAIN_DETECTOR:
    for src in det_manifest.source.unique():
        show_grid(det_manifest[det_manifest.source == src].image.tolist(), DETECT_CLASSES, title=src,
                  save=f"{DRIVE_ROOT}/reports/look_{src}.jpg")

## A3 · Scoring helpers

* `player_recall_per_frame`: players found / players labelled, averaged over frames
* `all_players_found_share`: frames where every labelled player was found
* `Ball_recall`, `Ball_precision`, `phantom_ball_frame_share` (frames with a false ball box)

Each source is scored only on the classes it labels itself, at the given thresholds.

In [ ]:
from courtvision.training.evaluate import (best_threshold, compare, mot_metrics, mot_table, predict_images,
                                           score_predictions, summarise, threshold_sweep)
from courtvision.training.tracking_eval import detect_sequence, track_players, tracking_sweep

KEY = ["frames", "player_recall_per_frame", "all_players_found_share", "players_found_per_frame", "Player_precision",
       "Ball_recall", "Ball_precision", "phantom_ball_frame_share", "Ref_recall", "Hoop_recall"]

def thresholds(player, ball):
    return {"Player": player, "Ref": player, "Hoop": player, "Ball": ball}

DEFAULT_CONF = thresholds(CFG.detection.conf, CFG.detection.ball_conf)

def det_images(split, sources=None, where=None):
    """{source: image paths} of one split; where: optional extra row filter on the manifest."""
    m = det_manifest[(det_manifest.split == split) & det_manifest.source.isin(list(sources or SCORED))]
    if where is not None:
        m = m[where.loc[m.index]]
    return {s: g.image.tolist() for s, g in m.groupby("source")}

def det_score(weights, by_source, conf=DEFAULT_CONF, imgsz=1280):
    """One row per source, scored on the classes that source labels itself."""
    model = YOLO(weights)
    rows = {}
    for s, imgs in by_source.items():
        if imgs:
            preds = predict_images(model, imgs, DETECT_CLASSES, imgsz, min_conf=min(conf.values()))
            rows[s] = summarise(score_predictions(preds, DETECT_CLASSES, {c: conf[c] for c in SCORED[s]}))
    return pd.DataFrame(rows).T.reindex(columns=KEY)

## A4 · Optional: which dataset matters (leave one dataset out)

For each training source, a small model trained **with** it and one trained **without** it (and
without anything tied to it), both scored on that source's test split. A big gap means that
dataset brings footage the others lack. Off unless `RUN_CROSS_VALIDATION`; the final model is the
same either way.

In [ ]:
from courtvision.training.sources import lodo_folds

if TRAIN_DETECTOR and RUN_CROSS_VALIDATION:
    CV = dict(imgsz=960, epochs=30, patience=10, batch=-1, cos_lr=True, close_mosaic=5, plots=False)
    folds = lodo_folds(DET_DATA, min_images=50)
    with_it = det_score(train(f"det_{RUN_TAG}_cv_all", f"{DET_DATA}/data.yaml", "yolo11s.pt", **CV),
                        det_images("test", list(folds)), imgsz=CV["imgsz"])
    without = pd.concat([det_score(train(f"det_{RUN_TAG}_cv_without_{held}", str(y), "yolo11s.pt", **CV),
                                   det_images("test", [held]), imgsz=CV["imgsz"]) for held, y in folds.items()])
    cols = ["player_recall_per_frame", "all_players_found_share", "Ball_recall", "phantom_ball_frame_share"]
    lodo = pd.concat({"with": with_it[cols], "without": without[cols]}, axis=1).swaplevel(axis=1)[cols]
    lodo.to_csv(f"{DRIVE_ROOT}/reports/detector_lodo_{RUN_TAG}.csv")
    display(lodo)

## A5 · Train the detector on everything

Broadcast balls are 15 to 30 px, so it trains at 1280. Early stopping uses the pooled validation
split; test is never touched.

In [ ]:
if TRAIN_DETECTOR:
    NEW_DET = train(f"det_{RUN_TAG}_final_{Path(DETECTOR['base']).stem}_{DETECTOR['imgsz']}", f"{DET_DATA}/data.yaml",
                    DETECTOR["base"], imgsz=DETECTOR["imgsz"], epochs=DETECTOR["epochs"], patience=DETECTOR["patience"],
                    batch=-1, cos_lr=True, close_mosaic=15, save_period=10)

## A6 · Tune the thresholds for tracking (validation data only)

Each detector, new and current, gets its own thresholds, so the comparison below is between the
two as the pipeline would really run them.

* **Players**: the pipeline's tracker (ByteTrack) starts tracks from confident boxes and uses
  low-confidence boxes only to keep an existing track going, through screens and crowds. So the
  player threshold is picked by **IDF1** (how much of each player's time stays under one id),
  tracking the validation SportsMOT clips at each threshold. Without SportsMOT it falls back to
  recall-weighted F2 on validation frames, with precision at least 0.85.
* **Ball**: F1 on validation frames of the sources that label the ball (a phantom ball moves the
  ball's court position as much as a missed one).

In [ ]:
P_GRID = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
B_GRID = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4]
TUNE_SEQS = 6   # validation SportsMOT clips, a few hundred frames each

def sm_games(split):
    return {Path(p).stem.split("__")[0] for p in det_images(split, ["sportsmot"]).get("sportsmot", [])}

def sm_sequences(split, n):
    return sportsmot.sequences_of_games(SM_DATA, sm_games(split))[:n] if SM_DATA and "sportsmot" in SCORED else []

def tune_detector(weights, label):
    """{"conf": players, "ball_conf": ball} for one detector, plus the sweeps behind them."""
    model = YOLO(weights)
    ball_imgs = [p for s, ims in det_images("valid").items() if "Ball" in SCORED[s] for p in ims]
    ball_sweep = threshold_sweep(predict_images(model, ball_imgs, DETECT_CLASSES, 1280, min_conf=min(B_GRID)),
                                 DETECT_CLASSES, "Ball", B_GRID, beta=1)
    ball = best_threshold(ball_sweep, "f1") if ball_imgs else CFG.detection.ball_conf
    seqs = sm_sequences("valid", TUNE_SEQS)
    if seqs:
        per_seq = {s.name: (detect_sequence(weights, s, CFG.detection, min_conf=min(P_GRID)), sportsmot.read_tracks(s),
                            sportsmot.seq_fps(s)) for s in seqs}
        player_sweep = tracking_sweep(per_seq, P_GRID, CFG.tracking.track_buffer)
        player = float(player_sweep.idf1.astype(float).idxmax())
    else:
        p_imgs = [p for s, ims in det_images("valid").items() if "Player" in SCORED[s] for p in ims]
        player_sweep = threshold_sweep(predict_images(model, p_imgs, DETECT_CLASSES, 1280, min_conf=min(P_GRID)),
                                       DETECT_CLASSES, "Player", P_GRID, beta=2)
        player = best_threshold(player_sweep, "f2", min_precision=0.85) if p_imgs else CFG.detection.conf
    print(f"{label}: players {player} (by {'IDF1 on ' + str(len(seqs)) + ' clips' if seqs else 'F2'}), ball {ball} (by F1)")
    display(pd.concat({"players": player_sweep, "ball": ball_sweep}, axis=1))
    return {"conf": player, "ball_conf": ball}

if TRAIN_DETECTOR:
    TUNED_DET = {"new": tune_detector(NEW_DET, "new")}
    if os.path.exists(CURRENT_DET):
        TUNED_DET["current"] = tune_detector(CURRENT_DET, "current")
    yaml.safe_dump(TUNED_DET, open(f"{DRIVE_ROOT}/reports/detector_thresholds_{RUN_TAG}.yaml", "w"))

## A7 · New detector vs current, per source (test data)

Each source's test split, each model at its own tuned thresholds. The current detector was trained
on v17 with v17's own random split, so it may have seen any v17 test image or a copy of one in
another dataset: the comparison that decides uses only test images not tied to any v17 image, and
the `v17` rows are for reference.

In [ ]:
if TRAIN_DETECTOR and os.path.exists(CURRENT_DET):
    v17_components = set(det_manifest[det_manifest.source == "v17"].component)
    unseen_by_current = ~det_manifest.component.isin(v17_components)
    DECIDE_ON = [s for s in ["own_test", "gamedet", "broadcast", "sportsmot", "own"] if s in SCORED]
    conf_of = {k: thresholds(v["conf"], v["ball_conf"]) for k, v in TUNED_DET.items()}
    fair = det_images("test", DECIDE_ON, where=unseen_by_current)
    det_table = pd.concat({"new": det_score(NEW_DET, fair, conf_of["new"]),
                           "current": det_score(CURRENT_DET, fair, conf_of["current"])}).swaplevel().sort_index()
    reference = pd.concat({"new": det_score(NEW_DET, det_images("test", ["v17"]), conf_of["new"]),
                           "current": det_score(CURRENT_DET, det_images("test", ["v17"]), conf_of["current"])})
    det_table.to_csv(f"{DRIVE_ROOT}/reports/detector_new_vs_current_{RUN_TAG}.csv")
    display(det_table)
    display(reference)

## A8 · Tracking on held-out SportsMOT games (test data)

Both detectors through the pipeline's tracker, at their tuned player thresholds, on the full 25 fps
clips of the SportsMOT games the build put in test. **MOTA** counts misses, false boxes and
identity switches against all labelled boxes; **IDF1** is how much of each player's time stays
under one id. Referees are left out on both sides (SportsMOT boxes only players).

In [ ]:
from dataclasses import replace

MAX_TEST_SEQS = 8
mot = {}
if TRAIN_DETECTOR:
    seqs = sm_sequences("test", MAX_TEST_SEQS)
    print(len(sm_games("test")) if seqs else 0, "test games; clips:", [s.name for s in seqs])
    for label, w in [("new", NEW_DET), ("current", CURRENT_DET)]:
        if seqs and label in TUNED_DET:
            mot[label] = mot_table({s.name: mot_metrics(sportsmot.read_tracks(s), track_players(
                detect_sequence(w, s, replace(CFG.detection, **TUNED_DET[label]), min_conf=TUNED_DET[label]["conf"]),
                TUNED_DET[label]["conf"],
                sportsmot.seq_fps(s), CFG.tracking.track_buffer)) for s in seqs})
    if mot:
        tracking = pd.concat({k: v.loc[["overall"]] for k, v in mot.items()})
        tracking.to_csv(f"{DRIVE_ROOT}/reports/detector_tracking_{RUN_TAG}.csv")
        display(tracking)

## A9 · Install the new detector only if it is better

On every source of the fair comparison the new model must find at least as many players per frame
and balls (within 0.01), without more phantom-ball frames, and track at least as well (IDF1,
MOTA). Whichever detector wins, its tuned thresholds go into `models/tuned_config.yaml`.

In [ ]:
def save_tuned(detection=None):
    t = (yaml.safe_load(open(TUNED)) if os.path.exists(TUNED) else None) or {}
    if detection:
        t.setdefault("detection", {}).update({k: float(v) for k, v in detection.items()})
    yaml.safe_dump(t, open(TUNED, "w"))
    print("tuned_config.yaml:", t)

if TRAIN_DETECTOR:
    verdict = {}
    if os.path.exists(CURRENT_DET):
        for src in det_table.index.get_level_values(0).unique():
            ok, why = compare(det_table.loc[(src, "new")], det_table.loc[(src, "current")],
                              higher=["player_recall_per_frame", "Ball_recall"], lower=["phantom_ball_frame_share"])
            verdict[src] = "ok" if ok else "; ".join(why)
        if len(mot) == 2:
            ok, why = compare(mot["new"].loc["overall"], mot["current"].loc["overall"], higher=["idf1", "mota"])
            verdict["tracking"] = "ok" if ok else "; ".join(why)
    for k, v in verdict.items():
        print(f"{k:12s} {v}")
    if any(v != "ok" for v in verdict.values()):
        print("\nKept the current detector. The new one is at", NEW_DET)
        save_tuned(TUNED_DET.get("current"))
    else:
        install(NEW_DET, CURRENT_DET, "detector")
        save_tuned(TUNED_DET["new"])

# Part B · Court keypoints

The court model finds up to 18 court landmarks in a frame; four or more give the mapping from the
image to the court in metres, which turns every player's feet into court x, y. The number that
matters most is **how often a real broadcast frame gets mapped**.

* `reloc2` is hand-labelled but only shows the whole court. Basketball Game Detection and
  Basketball Broadcast frames show the views the pipeline meets (half court, zoomed, panning), so
  they are **pseudo-labelled** by the current court model, kept only where a homography fits all
  its keypoints within 0.25 m, maps each back within 1% of the image width, and is the right way
  round (not a mirror image with the halves swapped). The fitted mapping then places every landmark
  in the frame, including the ones the model missed.
* Whole videos of the two broadcast sets are held back **before** pseudo-labelling and scored by
  mapping coverage (no labels needed). Landmark accuracy (PCK) is measured only on hand labels.
* Zoomed half-court crops of `reloc2` training images teach zoomed views.

Landmark order (`courtvision/court.py`): 0-5 left baseline (corner, corner-3, lane, lane, corner-3,
corner), 6-7 half-court line ends, 8-9 left free-throw line ends, 10-15 right baseline (mirrored),
16-17 right free-throw line ends.

## B1 · Check the landmark order on the hand labels

The numbers drawn on the court must match the order above. Real views all map the same way round,
so the share printed should be close to 100%; that orientation is what the checks below use.

In [ ]:
from collections import Counter
import cv2
from courtvision.court import FLIP_IDX, NBA
from courtvision.training.labels import label_orientation

if TRAIN_COURT:
    show_grid(glob.glob(f"{ROOTS['reloc2']}/*/images/*"), ["court"], title="reloc2 labels", n=3)
    signs = Counter(label_orientation(row, NBA) for lbl in glob.glob(f"{ROOTS['reloc2']}/*/labels/*.txt")
                    for row in open(lbl) if row.split())
    fitted = sum(v for k, v in signs.items() if k)
    SIGN = max((k for k in signs if k), key=signs.get)
    print(f"orientation {SIGN:+d} in {signs[SIGN] / fitted:.1%} of {fitted} labelled frames ({signs[None]} had under 4 landmarks)")
    assert signs[SIGN] / fitted > 0.9, "labels disagree on orientation: check the landmark order before training"

## B2 · Pseudo-label the broadcast frames

The printout says how many frames were labelled and why the others were rejected
(`too_few_keypoints`: zoomed past what the model sees; `high_error` or `outlier_keypoints`: a
landmark in the wrong place; `disagrees_with_geometry`; `mirrored`). Rejected frames are not used.

In [ ]:
from courtvision.detect import best_court_instance
from courtvision.training.labels import court_row_from_keypoints
from courtvision.training.sources import SPLITS, hold_out_groups, images_in

HOLDOUT_FRAC = 0.2
POSE_YAML = {"names": {0: "court"}, "kpt_shape": [18, 3]}

def pseudo_label(name, root, teacher):
    """Hold back whole videos (unlabelled, for scoring); label the rest where the geometry checks pass."""
    imgs = [p for s in SPLITS if (Path(root) / s / "images").exists() for p in images_in(Path(root) / s / "images")]
    held = hold_out_groups([p.stem for p in imgs], HOLDOUT_FRAC, seed=0)
    lab, hold = Path(f"/content/datasets/{name}_court"), Path(f"/content/datasets/{name}_holdout")
    shutil.rmtree(lab, ignore_errors=True); shutil.rmtree(hold, ignore_errors=True)
    for d in [lab / "train" / "images", lab / "train" / "labels", hold / "test" / "images"]:
        d.mkdir(parents=True)
    reasons = Counter()
    for img_path, h in zip(imgs, held):
        if h:
            shutil.copy(img_path, hold / "test" / "images" / img_path.name)
            continue
        img = cv2.imread(str(img_path))
        kps = best_court_instance(teacher.predict(img, imgsz=640, conf=0.25, verbose=False)[0])
        row, info = court_row_from_keypoints(kps, NBA, (img.shape[1], img.shape[0]), orientation_sign=SIGN)
        reasons[info["reason"]] += 1
        if row:
            shutil.copy(img_path, lab / "train" / "images" / img_path.name)
            (lab / "train" / "labels" / f"{img_path.stem}.txt").write_text(row + "\n")
    for d in [lab, hold]:
        yaml.safe_dump(POSE_YAML, open(d / "data.yaml", "w"))
    print(f"{name}: {int(held.sum())} frames held back; labelled {reasons['ok']} of {sum(reasons.values())}; {dict(reasons)}")
    return str(lab), str(hold)

PSEUDO_COURT = {}
if TRAIN_COURT and os.path.exists(CURRENT_COURT):
    court_teacher = YOLO(CURRENT_COURT)
    PSEUDO_COURT = {name: pseudo_label(name, ROOTS[name], court_teacher) for name in ["gamedet", "broadcast"]}
    # Landmarks should sit on the court lines, including ones the model missed.
    for name, (lab, hold) in PSEUDO_COURT.items():
        show_grid(glob.glob(f"{lab}/train/images/*"), ["court"], title=f"{name}: pseudo-labels",
                  save=f"{DRIVE_ROOT}/reports/look_{name}_court.jpg")

## B3 · One leakage-free court dataset, plus zoomed crops

Same build as the detector. `flip_idx` lets training flip images left-right with the landmarks
swapped.

In [ ]:
from courtvision.training.crops import add_zoomed_crops
from courtvision.training.sources import class_names

def court_source(name, root, **kw):
    """A one-class pose dataset, whatever its class is called."""
    return Source(name, root, rename={n: "court" for n in class_names(Path(root) / "data.yaml")}, **kw)

COURT_DATA = "/content/datasets/court"
if TRAIN_COURT:
    COURT_SOURCES = [court_source("reloc2", ROOTS["reloc2"])]
    for name, (lab, hold) in PSEUDO_COURT.items():
        COURT_SOURCES += [court_source(f"{name}_court", lab), court_source(f"{name}_holdout", hold, role="test")]
    if os.path.exists(f"{TD}/own_labelled_court/data.yaml"):
        COURT_SOURCES.append(court_source("own_court", f"{TD}/own_labelled_court", weight=3))
    if os.path.exists(f"{TD}/own_court_test/data.yaml"):
        COURT_SOURCES.append(court_source("own_court_test", f"{TD}/own_court_test", role="test"))

    court_manifest = build(COURT_SOURCES, ["court"], COURT_DATA, val_frac=0.15, test_frac=0.15, seed=0,
                           kpt_shape=[18, 3], flip_idx=FLIP_IDX)
    report = leakage_report(court_manifest)
    print(report)
    assert report["components_in_two_splits"] == report["groups_in_two_splits"] == report["test_source_images_not_in_test"] == 0
    print("zoomed crops added:", add_zoomed_crops(COURT_DATA, ["reloc2"], per_image=2))
    court_manifest = pd.read_csv(f"{COURT_DATA}/manifest.csv")
    court_manifest.to_csv(f"{DRIVE_ROOT}/reports/court_manifest_{RUN_TAG}.csv", index=False)
    show_grid(court_manifest[court_manifest.crop_of.notna()].image.tolist(), ["court"], title="zoomed crops", n=3)
    display(summary(court_manifest, ["court"]))

## B4 · Scoring helpers

* `mapped_share`: frames whose keypoints give a consistent, right-way-round mapping (no labels needed)
* `keypoints_per_frame`: confident landmarks per frame
* `median_px_error`, `pck_2pct` (labelled landmarks found within 2% of the image width): hand-labelled frames only

`reloc2` and `own_court` are scored on their test split, `gamedet` and `broadcast` on their
held-back videos, `own_court_test` on all of it.

In [ ]:
from courtvision.training.evaluate import court_scores

HAND_LABELLED = {"reloc2", "own_court", "own_court_test"}
AUG = dict(fliplr=0.5, mosaic=0.5, degrees=0, perspective=0.0005, scale=0.3)

def court_eval_images(where=None):
    """{name: images} to score on; where: optional extra row filter on the manifest."""
    m = court_manifest[court_manifest.crop_of.isna() & (where if where is not None else True)]
    sets = {"reloc2": m[(m.source == "reloc2") & (m.split == "test")],
            "own_court": m[(m.source == "own_court") & (m.split == "test")],
            "own_court_test": m[m.source == "own_court_test"]}
    sets.update({name: m[m.source == f"{name}_holdout"] for name in PSEUDO_COURT})
    return {k: v.image.tolist() for k, v in sets.items() if len(v)}

def court_score(weights, by_name, imgsz=640):
    model = YOLO(weights)
    rows = {}
    for name, imgs in by_name.items():
        r = court_scores(model, imgs, NBA, imgsz=imgsz, kp_conf=CFG.court.keypoint_conf, orientation_sign=SIGN)
        if name not in HAND_LABELLED:
            r["median_px_error"] = r["pck_2pct"] = None   # no hand labels there
        rows[name] = r
    return pd.DataFrame(rows).T

## B5 · Optional: which dataset matters (leave one dataset out)

If the model trained without the broadcast pseudo-labels maps far fewer held-back broadcast
frames, those labels are doing their job. Off unless `RUN_CROSS_VALIDATION`.

In [ ]:
if TRAIN_COURT and RUN_CROSS_VALIDATION:
    CV = dict(imgsz=640, epochs=60, patience=15, batch=-1, cos_lr=True, plots=False, **AUG)
    EVAL_OF = {"reloc2": "reloc2", "gamedet_court": "gamedet", "broadcast_court": "broadcast", "own_court": "own_court"}
    folds = lodo_folds(COURT_DATA, min_images=50)
    targets = court_eval_images()
    with_it = court_score(train(f"court_{RUN_TAG}_cv_all", f"{COURT_DATA}/data.yaml", "yolo11s-pose.pt", **CV), targets)
    rows = []
    for held, y in folds.items():
        best = train(f"court_{RUN_TAG}_cv_without_{held}", str(y), "yolo11s-pose.pt", **CV)
        name = EVAL_OF.get(held, held)
        if name in targets:
            rows.append(court_score(best, {name: targets[name]}))
    cols = ["mapped_share", "keypoints_per_frame", "pck_2pct"]
    lodo = pd.concat({"with": with_it[cols], "without": pd.concat(rows)[cols]}, axis=1).swaplevel(axis=1)[cols]
    lodo.to_csv(f"{DRIVE_ROOT}/reports/court_lodo_{RUN_TAG}.csv")
    display(lodo)

## B6 · Train the court model on everything

In [ ]:
if TRAIN_COURT:
    NEW_COURT = train(f"court_{RUN_TAG}_final_{Path(COURT['base']).stem}", f"{COURT_DATA}/data.yaml", COURT["base"],
                      imgsz=COURT["imgsz"], epochs=COURT["epochs"], patience=COURT["patience"], batch=-1, cos_lr=True,
                      save_period=10, **AUG)

## B7 · New court model vs current, and install it only if it is better

The current court model was trained on `reloc2` with its own random split, so the comparison that
decides uses frames not tied to any `reloc2` image; the `reloc2` row is a guard. On every set the
new model must map at least as many frames and find at least as many landmarks (within 0.01), be at
least as accurate on hand labels, and not lose more than 0.03 PCK on `reloc2`.

In [ ]:
if TRAIN_COURT:
    verdict = {}
    if os.path.exists(CURRENT_COURT):
        reloc2_components = set(court_manifest[court_manifest.source == "reloc2"].component)
        unseen_by_current = ~court_manifest.component.isin(reloc2_components)
        fair = {k: v for k, v in court_eval_images(unseen_by_current).items() if k != "reloc2"}
        court_table = pd.concat({"new": court_score(NEW_COURT, fair), "current": court_score(CURRENT_COURT, fair)}).swaplevel().sort_index()
        r2 = {"reloc2": court_eval_images()["reloc2"]}
        court_ref = pd.concat({"new": court_score(NEW_COURT, r2), "current": court_score(CURRENT_COURT, r2)})
        court_table.to_csv(f"{DRIVE_ROOT}/reports/court_new_vs_current_{RUN_TAG}.csv")
        display(court_table)
        display(court_ref)
        for name in court_table.index.get_level_values(0).unique():
            ok, why = compare(court_table.loc[(name, "new")], court_table.loc[(name, "current")],
                              higher=["mapped_share", "keypoints_per_frame", "pck_2pct"], lower=["median_px_error"])
            verdict[name] = "ok" if ok else "; ".join(why)
        ok, why = compare(court_ref.loc[("new", "reloc2")], court_ref.loc[("current", "reloc2")], higher=["pck_2pct"], tol=0.03)
        verdict["reloc2 (guard)"] = "ok" if ok else "; ".join(why)
    for k, v in verdict.items():
        print(f"{k:16s} {v}")
    if any(v != "ok" for v in verdict.values()):
        print("\nKept the current court model. The new one is at", NEW_COURT)
    else:
        install(NEW_COURT, CURRENT_COURT, "court model")

# Part C · The tracking dataset, before vs after

The pipeline (notebook 00) on the first `E2E_SECONDS` of up to `E2E_MAX_CLIPS` clips in
`MyDrive/courtvision/input_videos/`, once with the models and thresholds in place before this run
and once with what is installed now. Read it top to bottom, like notebook 00's diagnostics: a stage
can only be as good as the ones before it.

| Column | |
|---|---|
| `mapped_share` | frames with an image-to-court mapping |
| `players_detected`, `players_placed` | players per frame found, and placed on the court |
| `short_tracks` | share of player tracks under 1 s (lower: fewer broken tracks) |
| `players_per_sample`, `all_ten_share`, `ball_share` | the 10 Hz dataset: players per sample, samples with all ten, samples with the ball |

In [ ]:
import subprocess
from courtvision import pipeline

def run_clip(clip, det, court, tuned):
    cfg = load_cfg(tuned)
    cfg.models.detector, cfg.models.court_keypoints = det, court
    cfg.output.cache_dir, cfg.output.write_video = "/content/e2e_stubs", False
    d = pipeline.run(clip, cfg, write_outputs=False).diagnostics
    ds = d["dataset"]
    return {"mapped_share": d["court"]["mapped_share"], "players_detected": d["players"]["detected_per_frame"],
            "players_placed": d["players"]["placed_per_frame"], "short_tracks": d["players"]["tracks_under_1s_share"],
            "players_per_sample": ds.get("players_per_sample"), "all_ten_share": ds.get("all_players_share"),
            "ball_share": ds.get("ball_share")}

clips = sorted(glob.glob(f"{DRIVE_ROOT}/input_videos/*.mp4"))[:E2E_MAX_CLIPS]
shutil.rmtree("/content/e2e_stubs", ignore_errors=True)   # model passes are cached by file path, and a file may have just been replaced
before = (f"{BEFORE}/player_detector.pt", f"{BEFORE}/court_keypoints.pt", f"{BEFORE}/tuned_config.yaml")
after = (CURRENT_DET, CURRENT_COURT, TUNED)
rows = {}
os.makedirs("/content/e2e", exist_ok=True)
for clip in clips:
    short = f"/content/e2e/{Path(clip).stem}.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", clip, "-t", str(E2E_SECONDS), "-c", "copy", short], check=True)
    for label, (det, court, tuned) in [("before", before), ("after", after)]:
        if os.path.exists(det) and os.path.exists(court):
            rows[(Path(clip).stem, label)] = run_clip(short, det, court, tuned)
if rows:
    e2e = pd.DataFrame(rows).T
    e2e.to_csv(f"{DRIVE_ROOT}/reports/tracking_before_after_{RUN_TAG}.csv")
    display(e2e)
else:
    print("No clips in", f"{DRIVE_ROOT}/input_videos", "- add a broadcast clip to see the before/after check.")